#  Multi-Agent Systems with Google ADK



---

## Table of Contents

1. [Introduction to Google ADK](#1-introduction-to-google-adk)
2. [Setting Up the Environment](#2-setting-up-the-environment)
3. [Building Your First Multi-Agent System](#3-building-your-first-multi-agent-system)
4. [**BREAK**](#break-1)
5. [Multi-Agent Collaboration Patterns](#5-multi-agent-collaboration-patterns)
6. [Coordinator Pattern](#6-coordinator-pattern)
7. [**BREAK**](#break-2)
8. [Parallel Agent Execution](#8-parallel-agent-execution)
9. [Practical Applications and Best Practices](#9-practical-applications-and-best-practices) (20 mins)

---

## Learning Objectives

By the end of this session, you will be able to:

- Understand Google ADK architecture and benefits
- Build specialized agents with `LlmAgent`
- Implement agent collaboration using sub-agents
- Create coordinator-based multi-agent systems
- Design parallel and sequential agent workflows
- Apply multi-agent patterns to real-world problems

---

## 1. Introduction to Google ADK

### What is Google ADK?

**Google ADK (Agent Development Kit)** is an open-source, code-first Python toolkit for building, evaluating, and deploying sophisticated AI agents with flexibility and control.

### Key Features

**Code-First Approach:**
- Agent development feels like software development
- Full control over agent behavior
- Type-safe and testable

**Multi-Agent Architecture:**
- Hierarchical agent structures
- Coordinator/dispatcher patterns
- Flexible orchestration

**Optimized for Gemini:**
- Native integration with Google's Gemini models
- Works with Vertex AI
- Supports other LLM providers

**Rich Tool Ecosystem:**
- Built-in tools for common tasks
- Easy custom tool creation
- Agent-as-tool pattern

### Core Components

1. **`LlmAgent`:** The primary agent class powered by LLMs
2. **`SequentialAgent`:** Execute agents one after another
3. **`ParallelAgent`:** Execute multiple agents concurrently
4. **`LoopAgent`:** Iterative agent execution with conditions
5. **Sub-agents:** Hierarchical agent relationships
6. **Tools:** Functions agents can call
7. **State Management:** Session state for inter-agent communication

### Why Google ADK for Multi-Agent Systems?

1. **Clean Abstractions:** Well-designed APIs for agent composition
2. **Production-Ready:** Built by Google for enterprise use
3. **Flexible:** Supports multiple orchestration patterns
4. **Integrated:** Works seamlessly with Google Cloud ecosystem
5. **Extensible:** Easy to customize and extend

---

## 2. Setting Up the Environment

### Install Required Packages

In [1]:
# Install Google ADK and dependencies
!pip install -q google-adk
!pip install -q google-cloud-aiplatform
!pip install -q google-genai

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [2]:
import os
from google.colab import auth

# Authenticate with Google Cloud
auth.authenticate_user()

# Set your project ID and region
PROJECT_ID = "project-4b8caa1e-21b0-4bbe-b08"  # Change GCP Project ID
LOCATION = "us-central1"  # Change if needed

# Configure for Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"✅ Authenticated with project: {PROJECT_ID}")
print(f"✅ Using region: {LOCATION}")
print(f"✅ Vertex AI enabled")

✅ Authenticated with project: project-4b8caa1e-21b0-4bbe-b08
✅ Using region: us-central1
✅ Vertex AI enabled


In [3]:
# ============================================================================
# GOOGLE ADK CORE IMPORTS
# ============================================================================

# --- Agent Classes ---
# These classes are the building blocks for creating AI agents

from google.adk.agents import (
    LlmAgent,          # Primary agent class powered by LLMs (like Gemini)
                       # Usage: Create agents that can use tools, follow instructions

    SequentialAgent,   # Execute sub-agents one after another in sequence
                       # Usage: Build pipelines where output of one agent feeds into next
                       # Example: researcher → analyzer → summarizer

    ParallelAgent,     # Execute multiple sub-agents concurrently
                       # Usage: Run independent tasks simultaneously for efficiency
                       # Example: fact_checker + tone_analyzer + length_checker

    BaseAgent          # Base class for creating custom agents
                       # Usage: Extend this to build agents with custom logic
                       # Example: Conditional workflows, special orchestration
)

# --- Runners ---
# Runners execute agents and manage their lifecycle

from google.adk.runners import (
    InMemoryRunner,    # Runner with built-in in-memory session storage
                       # Usage: Quick setup for single-agent systems
                       # Note: Each instance has its own session service

    Runner             # Configurable runner that accepts external session service
                       # Usage: When you need to share sessions across multiple runners
                       # Example: Manual orchestration with multiple agents
)

# --- Session Services ---
# Session services manage conversation history and state

from google.adk.sessions import (
    InMemorySessionService  # In-memory session storage
                            # Usage: Create one instance and share across runners
                            # Example: shared_service = InMemorySessionService()
)

# --- Event System ---
# Events communicate agent actions and enable streaming responses

from google.adk.events import (
    Event,             # Represents an agent event (message, tool call, etc.)
                       # Contains: author, content, timestamp, actions
                       # Usage: Stream agent outputs, handle intermediate results

    EventActions       # Actions an event can trigger (escalate, transfer, etc.)
                       # Usage: Control agent flow (stop loop, transfer to sub-agent)
)

# --- Context ---
# InvocationContext provides runtime information to agents

from google.adk.agents.invocation_context import (
    InvocationContext  # Runtime context passed to agents during execution
                       # Contains: session, user_id, state, invocation_id
                       # Usage: Access session state, user info in custom agents
)

# --- Gemini Types ---
# Type definitions for Gemini API interactions

from google.genai import types
# Common types used:
#   - types.Content: Message content (user/agent messages)
#   - types.Part: Message part (text, function call, etc.)
#   - types.GenerateContentConfig: Model configuration (temperature, etc.)
#   - types.Tool: Tool definitions for agents

# --- Python Standard Library ---

from typing import AsyncGenerator
# AsyncGenerator: Type hint for async generator functions
# Usage: Type hints for streaming agent responses
# Example: async def stream() -> AsyncGenerator[Event, None]

import asyncio
# asyncio: Python's async/await framework
# Usage: Run async functions in Colab with await
# Example: await agent.run_async(...)

print("✅ All imports successful!")
print()
print("📚 Import Summary:")
print("   - Agent Types: LlmAgent, SequentialAgent, ParallelAgent, BaseAgent")
print("   - Runners: InMemoryRunner (built-in session), Runner (external session)")
print("   - Session Services: InMemorySessionService (shareable)")
print("   - Events: Event, EventActions (streaming & control)")
print("   - Context: InvocationContext (runtime info)")
print("   - Types: Gemini API type definitions")
print("   - Async: AsyncGenerator, asyncio (for async execution)")

✅ All imports successful!

📚 Import Summary:
   - Agent Types: LlmAgent, SequentialAgent, ParallelAgent, BaseAgent
   - Runners: InMemoryRunner (built-in session), Runner (external session)
   - Session Services: InMemorySessionService (shareable)
   - Events: Event, EventActions (streaming & control)
   - Context: InvocationContext (runtime info)
   - Types: Gemini API type definitions
   - Async: AsyncGenerator, asyncio (for async execution)


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


---

## 3. Building Your First Multi-Agent System

Let's build a simple multi-agent system with specialized agents:
1. **Math Agent:** Handles calculations
2. **Research Agent:** Provides information

### Step 1: Define Tools for Agents

In ADK, tools are Python functions that agents can call.

In [4]:
# Math Tools
def add_numbers(a: float, b: float) -> str:
    """Add two numbers together.

    Args:
        a: First number
        b: Second number

    Returns:
        Sum of a and b
    """
    result = a + b
    print(f"🔢 Adding {a} + {b} = {result}")
    return f"The sum is {result}"

def multiply_numbers(a: float, b: float) -> str:
    """Multiply two numbers.

    Args:
        a: First number
        b: Second number

    Returns:
        Product of a and b
    """
    result = a * b
    print(f"🔢 Multiplying {a} × {b} = {result}")
    return f"The product is {result}"

# Research Tools
def search_knowledge(query: str) -> str:
    """Search knowledge base for information.

    Args:
        query: Search query

    Returns:
        Information from knowledge base
    """
    # Simulated knowledge base
    kb = {
        "python": "Python is a high-level programming language known for readability.",
        "ai": "Artificial Intelligence is the simulation of human intelligence by machines.",
        "google adk": "Google ADK is an open-source toolkit for building AI agents.",
        "vertex ai": "Vertex AI is Google Cloud's unified ML platform.",
    }

    query_lower = query.lower()
    for key, value in kb.items():
        if key in query_lower:
            print(f"🔍 Found: {key}")
            return value

    return f"No information found for '{query}'"

print("✅ Tools defined successfully!")

✅ Tools defined successfully!


### Step 2: Create Specialized LlmAgents

Each agent has:
- **name:** Unique identifier
- **model:** LLM to use (Gemini 2.5 Flash)
- **instruction:** What the agent should do
- **tools:** Functions the agent can call
- **output_key:** Where to save results in session state

In [5]:
# Create Math Agent
math_agent = LlmAgent(
    name="math_agent",
    model="gemini-2.5-flash",
    instruction="""You are a mathematical expert.

    Your responsibilities:
    - Perform calculations using the available tools
    - Use ONE tool at a time
    - Always use tools for calculations, never compute mentally
    - Explain your calculations clearly

    You only handle math. For other topics, say you cannot help.
    """,
    description="Handles mathematical calculations and operations.",
    tools=[add_numbers, multiply_numbers]
)
